In [10]:
!pip install Faker

1. Linguistically grounded Indonesian informal language (not mechanical noise)
2. Expanded template diversity (120 patterns)
3. Realistic transformations at sentence level (not just entity-level)
4. Validation functions for ecological validity

In [11]:
import re
import random
import string
from datetime import datetime
from typing import Optional, List, Dict, Tuple
import numpy as np
import pandas as pd
from faker import Faker
from tqdm import tqdm

In [12]:
SEED = 42
NUM_IDENTITIES = 5_000
EVAL_SAMPLE_SIZE = 500

random.seed(SEED)
np.random.seed(SEED)
fake_id = Faker("id_ID")
Faker.seed(SEED)

AREA_CODES = [
    "317101", "317201", "317301", "317401", "327101", "327201",
    "327301", "317501", "317601", "310101", "310201", "310301",
    "310401", "310501", "327401", "320101", "320201", "320301",
    "320401", "330101", "330201", "330301", "340101", "340201",
    "350101", "350201", "360101", "360201", "370101", "510101",
    "510201", "510301", "610101", "710101", "710201", "730101",
]

LINGUISTICALLY GROUNDED INDONESIAN INFORMAL TRANSFORMATIONS - Real phonological, lexical, and code-switching patterns

In [13]:
# Phonological reductions (common in Indonesian informal speech)
INFORMAL_WORD_TRANSFORMS = {
    # Common verbs
    "tolong": ["tlg", "tlong", "tolongin"],
    "mohon": ["mhn", "mohonin"],
    "bantu": ["bantuin", "bntu"],
    "cek": ["check", "cek", "cekkin"],
    "proses": ["proses", "prosesin"],
    "verifikasi": ["verif", "verify"],
    "update": ["update", "apdet"],
    "kirim": ["kirimin", "send"],

    # Common adjectives/adverbs
    "segera": ["asap", "cepet", "cepat", "cpt"],
    "urgent": ["urgent", "penting bgt", "mendesak", "buru"],
    "benar": ["bener", "betul", "bnr"],
    "salah": ["salah", "wrong", "slh", "ngaco", "ngawur"],
    "valid": ["valid", "sah"],

    # Pronouns (critical for informal Indonesian)
    "saya": ["gw", "gue", "aku", "sy", "gua"],
    "anda": ["lo", "lu", "kamu", "km"],
    "dia": ["dia", "dy"],
    "kami": ["kita", "kt"],
    "mereka": ["mrk", "mereka", "mreka"],

    # Common words
    "dengan": ["dgn", "sama", "ama", "sm"],
    "untuk": ["utk", "buat", "bt", "bwt"],
    "dari": ["dr", "dari"],
    "yang": ["yg", "yng"],
    "tidak": ["gak", "ga", "nggak", "ngga", "gx", "g", "ngak"],
    "sudah": ["udah", "udh", "dah", "dh"],
    "belum": ["blm", "belom", "blum"],
    "gimana": ["gmn", "gmana", "bagaimana"],
    "kenapa": ["knp", "knapa", "napa", "nape"],
    "sama": ["sm", "sama", "sma"],
    "juga": ["jg", "juga", "jga"],
    "bisa": ["bs", "bisa", "bsa"],
    "harus": ["hrs", "mesti"],
    "akan": ["akn", "bakal", "bkl"],
    "ini": ["ni", "ini"],
    "itu": ["tu", "itu"],
    "ada": ["ad", "ada"],
    "atau": ["ato", "atau"],
    "dan": ["dan", "n"],
    "tapi": ["tp", "tapi", "but", "tpi"],
    "kalau": ["klo", "kalo", "kl"],
    "karena": ["krn", "soalnya", "coz", "cus", "bcs"],
    "makasih": ["makasi", "thx", "thanks", "tq", "tengkyu", "mksh"],
    "terima kasih": ["makasi", "thanks", "thx", "tq", "mksh"],
    "maaf": ["maap", "sori", "sorry"],
    "silakan": ["silahkan", "monggo", "slkan"],
}

# Informal particles (critical for Indonesian authenticity)
INFORMAL_PARTICLES = ["dong", "deh", "sih", "kok", "lah", "kan", "ya", "yah", "nih", "tuh"]

# Code-switching patterns (English-Indonesian mixing)
CODE_SWITCH_PHRASES = {
    "mohon bantuannya": ["please help", "tolong dong", "help me"],
    "segera": ["ASAP", "asap", "urgent"],
    "terima kasih": ["thanks", "thank you", "thx"],
    "data": ["data", "info"],
    "nomor": ["number", "no", "nomor"],
    "rekening": ["account", "rekening", "rek"],
    "kartu kredit": ["credit card", "CC", "kartu kredit"],
}

# Typos and abbreviations (realistic, not mechanical)
COMMON_TYPOS = {
    "harap": ["hrp", "harap"],
    "sesuai": ["sesuai", "ssuai"],
    "terlampir": ["terlampir", "terlmpir"],
    "berikut": ["berikut", "brikut"],
}

PII GENERATION FUNCTIONS

In [14]:
def generate_nik(gender: str = "M", dob: Optional[datetime] = None) -> str:
    """Generate valid Indonesian NIK (16 digits)"""
    area_code = random.choice(AREA_CODES)
    if dob is None:
        dob = fake_id.date_of_birth(minimum_age=17, maximum_age=70)
    day = dob.day + (40 if gender == "F" else 0)
    dob_str = f"{day:02d}{dob.month:02d}{str(dob.year)[-2:]}"
    sequence = str(random.randint(1, 9999)).zfill(4)
    return f"{area_code}{dob_str}{sequence}"


def generate_indonesian_phone() -> str:
    """Generate realistic Indonesian phone number"""
    prefixes = [
        "0811", "0812", "0813", "0821", "0822", "0823", "0851", "0852", "0853",
        "0814", "0815", "0816", "0855", "0856", "0857", "0858",
        "0817", "0818", "0819", "0859", "0877", "0878",
        "0831", "0832", "0833", "0838",
        "0895", "0896", "0897", "0898", "0899",
        "0881", "0882", "0883", "0884", "0885",
    ]
    prefix = random.choice(prefixes)
    suffix_len = random.choice([7, 8])
    suffix = "".join([str(random.randint(0, 9)) for _ in range(suffix_len)])
    return f"+62{prefix[1:]}{suffix}" if random.random() < 0.3 else f"{prefix}{suffix}"


def generate_bank_account() -> str:
    """Generate bank account number (8-15 digits)"""
    length = random.randint(8, 15)
    return str(random.randint(1, 9)) + "".join([str(random.randint(0, 9)) for _ in range(length - 1)])


def generate_credit_card() -> str:
    """Generate valid credit card number with Luhn checksum"""
    prefixes = ["4", "51", "52", "53", "54", "55", "6011"]
    prefix = random.choice(prefixes)
    partial = prefix + "".join([str(random.randint(0, 9)) for _ in range(16 - len(prefix) - 1)])
    digits = [int(d) for d in partial]
    for i in range(len(digits) - 2, -1, -2):
        digits[i] *= 2
        if digits[i] > 9:
            digits[i] -= 9
    checksum = (10 - (sum(digits) % 10)) % 10
    return partial + str(checksum)


def generate_email_indonesian(name: str) -> str:
    """Generate Indonesian-style email"""
    domains = ["gmail.com", "yahoo.co.id", "hotmail.com", "outlook.com", "ymail.com"]
    clean_name = name.lower().replace(" ", random.choice([".", "_", ""]))
    suffix = random.choice(["", str(random.randint(1, 999))])
    return f"{clean_name}{suffix}@{random.choice(domains)}"


def generate_decoy_id():
    """16-digit string with invalid NIK prefix (distractor)"""
    return f"99{random.randint(10000000000000, 99999999999999)}"


def build_ground_truth_dataframe(n: int = NUM_IDENTITIES) -> pd.DataFrame:
    """Generate synthetic identities with PII"""
    print(f"Generating {n} synthetic identities...")
    records = []
    for i in tqdm(range(n), desc="Generating identities"):
        gender = random.choice(["M", "F"])
        dob = fake_id.date_of_birth(minimum_age=17, maximum_age=70)
        name = fake_id.name_male() if gender == "M" else fake_id.name_female()
        records.append({
            "id": i,
            "name": name,
            "gender": gender,
            "dob": dob,
            "nik": str(generate_nik(gender, dob)),
            "phone": str(generate_indonesian_phone()),
            "bank_account": str(generate_bank_account()),
            "credit_card": str(generate_credit_card()),
            "email": generate_email_indonesian(name),
            "address": fake_id.address().replace("\n", ", "),
            "decoy_id": str(generate_decoy_id()),
        })
    df = pd.DataFrame(records)
    df["nik"] = df["nik"].astype(str)
    df["phone"] = df["phone"].astype(str)
    df["bank_account"] = df["bank_account"].astype(str)
    df["credit_card"] = df["credit_card"].astype(str)
    df["decoy_id"] = df["decoy_id"].astype(str)
    return df

LINGUISTICALLY GROUNDED TRANSFORMATION FUNCTIONS - realistic Indonesian informal language patterns

In [15]:
def apply_phonological_reduction(text: str, intensity: float = 0.3) -> str:
    words = text.split()
    transformed_words = []

    for word in words:
        word_lower = word.lower()
        # Check if word matches any formal form
        transformed = False
        for formal, informal_variants in INFORMAL_WORD_TRANSFORMS.items():
            if formal in word_lower and random.random() < intensity:
                # Preserve capitalization pattern
                if word[0].isupper():
                    replacement = random.choice(informal_variants).capitalize()
                else:
                    replacement = random.choice(informal_variants)
                transformed_words.append(word.replace(formal, replacement))
                transformed = True
                break

        if not transformed:
            transformed_words.append(word)

    return " ".join(transformed_words)


def add_informal_particles(text: str, intensity: float = 0.2) -> str:
    """
    Add Indonesian informal particles (dong, deh, sih, etc.)
    """
    sentences = text.split(".")
    transformed_sentences = []

    for sentence in sentences:
        sentence = sentence.strip()
        if sentence and random.random() < intensity:
            # Add particle at end of sentence
            particle = random.choice(INFORMAL_PARTICLES)
            sentence = f"{sentence} {particle}"
        transformed_sentences.append(sentence)

    return ". ".join(transformed_sentences)


def apply_code_switching(text: str, intensity: float = 0.3) -> str:
    """
    Apply English-Indonesian code-switching patterns. Common in urban Indonesian communication, especially in professional contexts.
    """
    for indonesian, english_variants in CODE_SWITCH_PHRASES.items():
        if indonesian in text.lower() and random.random() < intensity:
            replacement = random.choice(english_variants)
            text = re.sub(indonesian, replacement, text, flags=re.IGNORECASE)

    return text


def add_informal_punctuation(text: str, intensity: float = 0.3) -> str:
    """
    Add informal punctuation patterns (multiple punctuation, emoticons). Common in Indonesian social media and messaging.
    """
    if random.random() < intensity:
        # Multiple punctuation
        text = text.replace("!", random.choice(["!!", "!!!", "!"]))
        text = text.replace("?", random.choice(["??", "???", "?"]))
        # Add emoticons occasionally
        if random.random() < 0.2:
            emoticons = ["wkwk", "wkwkwk", "haha", "hehe", "lol", "lmao", "lmfao", "akowkw", "ahaha", "ehehhe"]
            text += " " + random.choice(emoticons)

    return text


def apply_abbreviations(text: str, intensity: float = 0.3) -> str:
    """
    Apply common Indonesian abbreviations and shortenings.
    """
    # Remove vowels from some words (common in Indonesian texting)
    words = text.split()
    transformed_words = []

    for word in words:
        if len(word) > 4 and random.random() < intensity * 0.5:
            # Remove some vowels (but keep at least one)
            vowels = "aiueo"
            new_word = ""
            vowel_count = sum(1 for c in word.lower() if c in vowels)

            if vowel_count > 1:
                for char in word:
                    if char.lower() in vowels and random.random() < 0.4:
                        continue  # Skip this vowel
                    new_word += char

                if len(new_word) >= 2:  # Ensure word is still readable
                    transformed_words.append(new_word)
                else:
                    transformed_words.append(word)
            else:
                transformed_words.append(word)
        else:
            transformed_words.append(word)

    return " ".join(transformed_words)


def apply_informal_transformations(text: str, style: str) -> str:
    if style == "formal":
        # No transformations for formal style
        return text

    elif style == "code_mixed":
        # Moderate transformations: code-switching + light phonological reduction
        text = apply_code_switching(text, intensity=0.4)
        text = apply_phonological_reduction(text, intensity=0.2)
        text = add_informal_particles(text, intensity=0.1)
        return text

    elif style == "informal":
        # Heavy transformations: all features
        text = apply_phonological_reduction(text, intensity=0.5)
        text = add_informal_particles(text, intensity=0.3)
        text = apply_code_switching(text, intensity=0.3)
        text = apply_abbreviations(text, intensity=0.3)
        text = add_informal_punctuation(text, intensity=0.4)
        return text

    return text

ENTITY-LEVEL NOISE (OPTIONAL, REALISTIC) - noise should be linguistically motivated

    Apply realistic noise to PII entities (optional, lower priority than linguistic transforms).
    
    Unlike the original inject_noise(), this applies contextually appropriate noise:
    - Spacing variations (common in manual typing)
    - Partial masking (privacy-conscious users)
    - Typos in specific positions (not mechanical OCR simulation)

In [16]:
def apply_realistic_entity_noise(text: str, pii_type: str, noise_level: float = 0.2) -> str:
    if random.random() > noise_level:
        return text

    noise_type = random.choice(["spacing", "partial_mask", "typo"])

    if noise_type == "spacing" and len(text) > 8:
        # Add spaces or dashes (common in manual entry)
        if pii_type in ["phone", "nik", "credit_card"]:
            # Group digits
            parts = [text[i:i+4] for i in range(0, len(text), 4)]
            separator = random.choice([" ", "-", ""])
            return separator.join(parts)

    elif noise_type == "partial_mask" and len(text) > 12:
        # Partial masking (privacy-conscious behavior)
        if pii_type in ["credit_card", "nik"]:
            mask_start = random.randint(8, 12)
            return text[:mask_start] + "X" * (len(text) - mask_start)

    elif noise_type == "typo" and len(text) > 4:
        # Single character typo (realistic human error)
        pos = random.randint(1, len(text) - 2)
        text_list = list(text)
        if text_list[pos].isdigit():
            # Adjacent digit typo
            text_list[pos] = str((int(text_list[pos]) + random.choice([-1, 1])) % 10)
        return "".join(text_list)

    return text

def generate_formal_templates(n: int = 40) -> List[str]:
    """Generate diverse formal Indonesian templates"""

    # Template components for variation
    greetings = [
        "Dengan hormat,",
        "Kepada Yth.,",
        "Bapak/Ibu,",
        "Yang terhormat,",
    ]

    actions = [
        "mohon bantuannya untuk memproses",
        "kami mengajukan permohonan verifikasi",
        "mohon lakukan pembaruan",
        "harap segera ditindaklanjuti",
        "kami sampaikan data untuk keperluan",
        "terlampir data untuk proses",
        "mohon verifikasi",
        "harap periksa",
    ]

    contexts = [
        "data nasabah berikut ini",
        "identitas pelanggan",
        "informasi KYC",
        "data untuk audit",
        "berkas administrasi",
        "dokumen verifikasi",
    ]

    closings = [
        "Harap segera ditindaklanjuti.",
        "Terima kasih atas perhatiannya.",
        "Atas perhatiannya kami ucapkan terima kasih.",
        "Demikian kami sampaikan.",
    ]

    templates = []

    # Generate varied combinations
    for _ in range(n):
        # Randomly select components
        greeting = random.choice(greetings) if random.random() < 0.6 else ""
        action = random.choice(actions)
        context = random.choice(contexts)
        closing = random.choice(closings) if random.random() < 0.7 else ""

        pii_fields = []
        if random.random() < 0.9:
            pii_fields.append("Nama: {name}")
        if random.random() < 0.9:
            pii_fields.append("NIK: {nik}")
        if random.random() < 0.7:
            pii_fields.append(random.choice(["Nomor Telepon: {phone}", "HP: {phone}", "Kontak: {phone}"]))
        if random.random() < 0.6:
            pii_fields.append(random.choice(["Nomor Rekening: {bank_account}", "Rek: {bank_account}"]))
        if random.random() < 0.7:
            pii_fields.append(random.choice(["Kartu Kredit: {credit_card}", "CC: {credit_card}"]))
        if random.random() < 0.7:
            pii_fields.append("Email: {email}")

        # Construct template
        pii_str = ", ".join(pii_fields)
        parts = [p for p in [greeting, f"{action} {context}.", pii_str, closing] if p]
        template = " ".join(parts)

        templates.append(template)

    return templates


def generate_code_mixed_templates(n: int = 40) -> List[str]:
    """Generate diverse code-mixed (English-Indonesian) templates"""

    greetings = [
        "Hi team,",
        "Halo guys,",
        "Hey everyone,",
        "Guys,",
        "FYI team,",
    ]

    actions = [
        "tolong help me check",
        "please update",
        "bisa tolong verify",
        "mohon recheck",
        "tolong process",
        "help me validate",
    ]

    urgency = [
        "ASAP ya",
        "urgent nih",
        "segera dong",
        "cepet ya",
        "deadline today",
    ]

    closings = [
        "Thanks!",
        "Makasih ya!",
        "Thx banget!",
        "Waiting for your response!",
    ]

    templates = []

    for _ in range(n):
        greeting = random.choice(greetings) if random.random() < 0.7 else ""
        action = random.choice(actions)
        urgency_phrase = random.choice(urgency) if random.random() < 0.6 else ""
        closing = random.choice(closings) if random.random() < 0.7 else ""

        pii_fields = []
        if random.random() < 0.9:
            pii_fields.append(random.choice(["user {name}", "nama {name}", "{name}"]))
        if random.random() < 0.9:
            pii_fields.append(random.choice(["NIK {nik}", "NIKnya {nik}", "ID {nik}"]))
        if random.random() < 0.7:
            pii_fields.append(random.choice(["phone {phone}", "HP {phone}", "nomer {phone}"]))
        if random.random() < 0.6:
            pii_fields.append(random.choice(["account {bank_account}", "rekening {bank_account}"]))
        if random.random() < 0.7:
            pii_fields.append(random.choice(["CC {credit_card}", "credit card {credit_card}"]))
        if random.random() < 0.7:
            pii_fields.append(random.choice(["email {email}", "Email: {email}"]))

        pii_str = ", ".join(pii_fields)
        parts = [p for p in [greeting, action, pii_str, urgency_phrase, closing] if p]
        template = " ".join(parts)

        templates.append(template)

    return templates


def generate_informal_templates(n: int = 40) -> List[str]:
    """Generate diverse informal Indonesian templates (will be further transformed)"""

    greetings = [
        "Halo",
        "Hai",
        "Weh",
        "Ges",
        "Bro",
        "Guys",
    ]

    actions = [
        "tolong bantu",
        "bantuin dong",
        "cek",
        "proses",
        "verif",
        "update",
    ]

    contexts = [
        "data ini",
        "info user",
        "data nasabah",
        "berkas",
    ]

    templates = []

    for _ in range(n):
        greeting = random.choice(greetings) if random.random() < 0.8 else ""
        action = random.choice(actions)
        context = random.choice(contexts) if random.random() < 0.6 else ""

        pii_fields = []
        if random.random() < 0.9:
            pii_fields.append(random.choice(["nama {name}", "{name}", "an {name}"]))
        if random.random() < 0.9:
            pii_fields.append(random.choice(["NIK {nik}", "KTP {nik}", "nik {nik}"]))
        if random.random() < 0.7:
            pii_fields.append(random.choice(["hp {phone}", "nomer {phone}", "telp {phone}"]))
        if random.random() < 0.6:
            pii_fields.append(random.choice(["rek {bank_account}", "rekening {bank_account}"]))
        if random.random() < 0.7:
            pii_fields.append(random.choice(["CC {credit_card}", "kartu kredit {credit_card}"]))
        if random.random() < 0.7:
            pii_fields.append(random.choice(["email {email}", "emailnya {email}"]))

        pii_str = " ".join(pii_fields)
        parts = [p for p in [greeting, action, context, pii_str] if p]
        template = " ".join(parts)

        templates.append(template)

    return templates


MAIN DATASET GENERATION FUNCTION

In [17]:
def wrap_pii_in_prompt(row: pd.Series, style: str, templates: Dict[str, List[str]]) -> dict:
    template = random.choice(templates[style])

    # Apply realistic entity-level noise (optional, low intensity)
    noise_level = {"formal": 0.0, "code_mixed": 0.05, "informal": 0.15}[style]

    nik_val = apply_realistic_entity_noise(str(row["nik"]), "nik", noise_level)
    phone_val = apply_realistic_entity_noise(str(row["phone"]), "phone", noise_level)
    cc_val = apply_realistic_entity_noise(str(row["credit_card"]), "credit_card", noise_level)
    bank_val = apply_realistic_entity_noise(str(row["bank_account"]), "bank_account", noise_level)

    # Format template with PII
    prompt = template.format(
        name=row["name"],
        nik=nik_val,
        phone=phone_val,
        credit_card=cc_val,
        bank_account=bank_val,
        email=row["email"],
        decoy_id=row["decoy_id"],
    )

    # Apply linguistically grounded transformations to entire sentence
    prompt = apply_informal_transformations(prompt, style)

    # Track which fields are used
    fields_used = {
        "nik": "{nik}" in template,
        "phone": "{phone}" in template,
        "credit_card": "{credit_card}" in template,
        "bank_account": "{bank_account}" in template,
        "email": "{email}" in template,
    }

    return {
        "identity_id": row["id"],
        "style": style,
        "prompt": prompt,
        "ground_truth_nik": str(row["nik"]) if fields_used["nik"] else None,
        "ground_truth_phone": str(row["phone"]) if fields_used["phone"] else None,
        "ground_truth_cc": str(row["credit_card"]) if fields_used["credit_card"] else None,
        "ground_truth_bank": str(row["bank_account"]) if fields_used["bank_account"] else None,
        "ground_truth_email": row["email"] if fields_used["email"] else None,
        "has_nik": fields_used["nik"],
        "has_phone": fields_used["phone"],
        "has_cc": fields_used["credit_card"],
        "has_bank": fields_used["bank_account"],
        "has_email": fields_used["email"],
    }


def build_prompt_dataset(df_truth: pd.DataFrame) -> pd.DataFrame:
    """
    Build full prompt dataset with expanded templates and linguistic transformations.

    Addresses MW1 and MW2:
    - 120 total templates (40 per style) vs original 15
    - Linguistically grounded transformations
    - Realistic Indonesian informal language
    """
    print("Generating expanded template set...")
    templates = {
        "formal": generate_formal_templates(40),
        "code_mixed": generate_code_mixed_templates(40),
        "informal": generate_informal_templates(40),
    }

    print(f"Generated {sum(len(t) for t in templates.values())} templates (40 per style)")
    print("Building contextual prompt dataset with linguistic transformations...")

    all_wrapped = []
    for style in ["formal", "code_mixed", "informal"]:
        for _, row in tqdm(df_truth.iterrows(), total=len(df_truth), desc=f"{style} style"):
            all_wrapped.append(wrap_pii_in_prompt(row, style, templates))

    df_prompts = pd.DataFrame(all_wrapped)

    cols_to_fix = ['ground_truth_nik', 'ground_truth_phone', 'ground_truth_cc', 'ground_truth_bank']
    for col in cols_to_fix:
        df_prompts[col] = df_prompts[col].astype(str).str.replace(r'\.0$', '', regex=True)
        df_prompts[col] = df_prompts[col].replace(['nan', 'None', 'NaN'], None)

    print(f"Prompt dataset shape: {df_prompts.shape}")
    return df_prompts


def get_eval_sample(df_prompts: pd.DataFrame, n: int = EVAL_SAMPLE_SIZE, seed: int = SEED) -> pd.DataFrame:
    frames = []
    for style in ["formal", "code_mixed", "informal"]:
        subset = df_prompts[df_prompts["style"] == style]
        frames.append(subset.sample(n=min(n, len(subset)), random_state=seed))

    sample = pd.concat(frames, ignore_index=True)
    print(f"Eval sample: {len(sample)} rows ({n}/style × 3 styles)")
    return sample


VALIDATION FUNCTIONS - ecological validation. Validate that informal samples contain authentic Indonesian linguistic features.  Returns DataFrame with feature counts for manual inspection.

In [23]:
def validate_linguistic_features(df_prompts: pd.DataFrame, sample_size: int = 100) -> pd.DataFrame:
    print("LINGUISTIC FEATURE VALIDATION")

    informal_sample = df_prompts[df_prompts["style"] == "informal"].sample(n=min(sample_size, len(df_prompts)))

    validation_results = []

    for _, row in informal_sample.iterrows():
        prompt = row["prompt"]

        # Count linguistic features
        features = {
            "has_informal_pronoun": any(p in prompt.lower() for p in ["gw", "gue", "lo", "lu", "aku"]),
            "has_particle": any(p in prompt.lower() for p in INFORMAL_PARTICLES),
            "has_phonological_reduction": any(
                informal in prompt.lower()
                for formals in INFORMAL_WORD_TRANSFORMS.values()
                for informal in formals
            ),
            "has_abbreviation": any(c in prompt for c in ["tlg", "dgn", "utk", "yg", "gx"]),
            "has_code_switching": any(eng in prompt.lower() for eng in ["please", "help", "check", "asap", "thanks"]),
            "prompt": prompt[:100] + "..." if len(prompt) > 100 else prompt,
        }

        validation_results.append(features)

    df_validation = pd.DataFrame(validation_results)

    # Print summary statistics
    print(f"\nValidation sample size: {len(df_validation)}")
    print(f"\nFeature prevalence in informal samples:")
    for col in df_validation.columns:
        if col.startswith("has_"):
            prevalence = df_validation[col].sum() / len(df_validation) * 100
            print(f"  {col}: {prevalence:.1f}%")

    print(f"\nSample prompts:")
    for i, row in df_validation.head(5).iterrows():
        print(f"\n  [{i+1}] {row['prompt']}")

    return df_validation


def compare_token_distributions(df_prompts: pd.DataFrame) -> Dict:
    """
    Compare token-level statistics across styles.  Helps verify that informal style has different distribution from formal.
    """
    print("TOKEN DISTRIBUTION ANALYSIS")
    stats = {}

    for style in ["formal", "code_mixed", "informal"]:
        subset = df_prompts[df_prompts["style"] == style]

        # Calculate statistics
        token_lengths = subset["prompt"].apply(lambda x: len(x.split()))
        char_lengths = subset["prompt"].apply(len)

        stats[style] = {
            "mean_tokens": token_lengths.mean(),
            "std_tokens": token_lengths.std(),
            "mean_chars": char_lengths.mean(),
            "std_chars": char_lengths.std(),
            "sample_size": len(subset),
        }

        print(f"\n{style.upper()} style:")
        print(f"  Mean tokens: {stats[style]['mean_tokens']:.1f} ± {stats[style]['std_tokens']:.1f}")
        print(f"  Mean chars: {stats[style]['mean_chars']:.1f} ± {stats[style]['std_chars']:.1f}")

    return stats


def export_validation_sample(df_prompts: pd.DataFrame, output_path: str = "validation_sample.csv"):
    sample = df_prompts.groupby("style").sample(n=50, random_state=SEED)
    sample[["style", "prompt"]].to_csv(output_path, index=False)
    print(f"\nValidation sample exported to: {output_path}")

In [24]:
if __name__ == "__main__":
    print("DLP DATASET GENERATION")

    # Generate identities
    df_truth = build_ground_truth_dataframe(NUM_IDENTITIES)
    print(f"\nGenerated {len(df_truth)} identities")

    # Build prompt dataset with expanded templates and linguistic transformations
    df_prompts = build_prompt_dataset(df_truth)

    # Validation
    print("RUNNING VALIDATION CHECKS")

    df_validation = validate_linguistic_features(df_prompts, sample_size=100)
    token_stats = compare_token_distributions(df_prompts)
    export_validation_sample(df_prompts)

    # Get evaluation sample
    df_eval = get_eval_sample(df_prompts, n=EVAL_SAMPLE_SIZE)

    # Save outputs
    df_truth.to_csv("ground_truth.csv", index=False)
    df_prompts.to_csv("prompt_dataset.csv", index=False)
    df_eval.to_csv("eval_sample.csv", index=False)
    df_validation.to_csv("linguistic_validation.csv", index=False)

    print("\nDataset generation complete!")
    print(f"  Ground truth: {len(df_truth)} identities")
    print(f"  Full dataset: {len(df_prompts)} prompts")
    print(f"  Eval sample: {len(df_eval)} prompts")
    print(f"  Templates: 120 total (40 per style)")
    print(f"  Styles: formal, code_mixed, informal")

DLP DATASET GENERATION
Generating 5000 synthetic identities...


Generating identities: 100%|██████████| 5000/5000 [00:00<00:00, 7736.26it/s]



Generated 5000 identities
Generating expanded template set...
Generated 120 templates (40 per style)
Building contextual prompt dataset with linguistic transformations...


informal style: 100%|██████████| 5000/5000 [00:00<00:00, 5167.51it/s]


Prompt dataset shape: (15000, 13)
RUNNING VALIDATION CHECKS
LINGUISTIC FEATURE VALIDATION

Validation sample size: 100

Feature prevalence in informal samples:
  has_informal_pronoun: 51.0%
  has_particle: 80.0%
  has_phonological_reduction: 100.0%
  has_abbreviation: 2.0%
  has_code_switching: 3.0%

Sample prompts:

  [1] Bro verif info user Karimah NapTupulu, S yah. Sos nik 3301012410891953 telp +6281676460333

  [2] cek nama Ella Namaga, S. Kom nik 3303012003970511 nomer +628981611560 kartu kredit 6011152020720763 ...

  [3] Weh prssn Lili Tampubolon, M. Kom. NIK 3272010106081186 rkening 241559388296 kartu krdit 4010-2291-6...

  [4] Weh tlong bantu info user Fitriani Pratama KTP 3204014401861538 rekening 1573510608 kartu kredit 601...

  [5] verif nama R. Tirtayasa Permta, S. Farm KTP 3175015112669449 nomer +6281268131086 kartu kredit 53729...
TOKEN DISTRIBUTION ANALYSIS

FORMAL style:
  Mean tokens: 19.9 ± 4.2
  Mean chars: 184.8 ± 42.8

CODE_MIXED style:
  Mean tokens: 20.7 ± 4.8

In [20]:
prompt = pd.read_csv("prompt_dataset.csv")
prompt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   identity_id         15000 non-null  int64  
 1   style               15000 non-null  object 
 2   prompt              15000 non-null  object 
 3   ground_truth_nik    12857 non-null  float64
 4   ground_truth_phone  10249 non-null  float64
 5   ground_truth_cc     10040 non-null  float64
 6   ground_truth_bank   9045 non-null   float64
 7   ground_truth_email  10975 non-null  object 
 8   has_nik             15000 non-null  bool   
 9   has_phone           15000 non-null  bool   
 10  has_cc              15000 non-null  bool   
 11  has_bank            15000 non-null  bool   
 12  has_email           15000 non-null  bool   
dtypes: bool(5), float64(4), int64(1), object(3)
memory usage: 1010.9+ KB


In [21]:
prompt.head(3)

,identity_id,style,prompt,ground_truth_nik,ground_truth_phone,ground_truth_cc,ground_truth_bank,ground_truth_email,has_nik,has_phone,has_cc,has_bank,has_email
0,0,formal,mohon bantuannya untuk memproses data nasabah ...,3.172011e+15,6.285822e+11,5.202654e+15,4.890839e+10,opung_saefullah619@gmail.com,True,True,True,True,True
1,1,formal,"Kepada Yth., mohon bantuannya untuk memproses ...",3.501010e+15,8.859593e+10,5.428328e+15,2.647526e+10,NaN,True,True,True,True,False
2,2,formal,mohon bantuannya untuk memproses identitas pel...,3.105014e+15,8.382424e+10,NaN,6.328710e+13,"dt..omar.haryanti,.m.pd@hotmail.com",True,True,False,True,True


In [22]:
gt = pd.read_csv("ground_truth.csv")
gt.head(3)

,id,name,gender,dob,nik,phone,bank_account,credit_card,email,address,decoy_id
0,0,Opung Saefullah,M,1989-11-08,3172010811894507,628582181960,48908386379,5202654235116152,opung_saefullah619@gmail.com,"Jl. Tebet Barat Dalam No. 8, Semarang, Sulawes...",9974659571848144
1,1,Sutan Nasrullah Yulianti,M,1987-11-01,3501010111871292,88595931034,26475255341,5428327648350300,sutan_nasrullah_yulianti411@gmail.com,"Gang Abdul Muis No. 654, Jambi, NB 16155",9989826016962532
2,2,"Dt. Omar Haryanti, M.Pd",F,1987-12-04,3105014412878180,83824238849,63287101226916,5397848018451463,"dt..omar.haryanti,.m.pd@hotmail.com","Gg. Medokan Ayu No. 5, Tarakan, DI Yogyakarta ...",9980454522783883
